In [ ]:
!pip install librosa lightgbm optuna pyloudnorm --quiet

In [ ]:
import os
import warnings
import pickle
import numpy as np
import pandas as pd
import librosa
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal as sig
from collections import Counter
from scipy.stats import kurtosis, skew

import pyloudnorm as pyln
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score)
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score)
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
SAMPLE_RATE = 16000
CLIP_DURATION = 3
TARGET_LEN = SAMPLE_RATE * CLIP_DURATION
N_MFCC = 13
N_MELS = 40
N_MFCC_QUARTERS = 4
TARGET_PER_CLASS = 1200
MIN_CYCLE_SECS = 0.5
TARGET_LUFS = -23.0
PHONE_AUG_PROB = 0.30

In [ ]:
print(f"Target clip length: {TARGET_LEN} samples ({CLIP_DURATION}s @ {SAMPLE_RATE}Hz)")
print(f"Phone aug probability: {PHONE_AUG_PROB*100:.0f}%")

In [ ]:
path1 = kagglehub.dataset_download("vbookshelf/respiratory-sound-database")
path2 = kagglehub.dataset_download("tareqkhanemu/snoring")
path3 = kagglehub.dataset_download("sylkaladin/speech-commands-v2")
print("ICBHI path  :", path1)
print("Snoring path:", path2)
print("Speech path :", path3)

ICBHI_PATH   = os.path.join(
    path1, "Respiratory_Sound_Database",
    "Respiratory_Sound_Database", "audio_and_txt_files"
)
SNORING_PATH = os.path.join(path2, "Snoring Dataset")
OTHER_PATH   = path3

In [ ]:
wav_count   = len([f for f in os.listdir(ICBHI_PATH)                   if f.endswith('.wav')])
snore_count = len([f for f in os.listdir(os.path.join(SNORING_PATH, '1')) if f.endswith('.wav')])
print(f"\nICBHI wavs: {wav_count}")
print(f"Snore clips: {snore_count}")

In [ ]:
_meter = pyln.Meter(SAMPLE_RATE)

In [ ]:
def normalize_loudness(audio: np.ndarray) -> np.ndarray:
    audio64  = audio.astype(np.float64)
    loudness = _meter.integrated_loudness(audio64)
    if not (np.isinf(loudness) or np.isnan(loudness)):
        audio64 = pyln.normalize.loudness(audio64, loudness, TARGET_LUFS)
    return np.clip(audio64, -1.0, 1.0).astype(np.float32)

In [ ]:
def prepare_clip(audio: np.ndarray, target_len: int = TARGET_LEN) -> np.ndarray:
    if len(audio) >= target_len:
        return audio[:target_len].copy()
    return np.pad(audio, (0, target_len - len(audio)))

In [ ]:
def simulate_phone_recording(audio: np.ndarray,
                              sr: int = SAMPLE_RATE) -> np.ndarray:

    audio = audio.copy().astype(np.float64)
    n = len(audio)

    # High-pass roll-off 
    nyq      = sr / 2.0
    hp_freq  = np.random.uniform(60, 200) / nyq
    hp_freq  = min(hp_freq, 0.999)
    b, a     = sig.butter(1, hp_freq, btype='high')
    audio    = sig.filtfilt(b, a, audio)

    # Ambient noise
    snr_db    = np.random.uniform(10, 25)
    sig_rms   = np.sqrt(np.mean(audio ** 2)) + 1e-9
    noise_rms = sig_rms / (10 ** (snr_db / 20.0))
    audio     = audio + np.random.normal(0, noise_rms, n)

    # Mild reverb
    reverb_ms  = np.random.uniform(40, 150)
    reverb_len = int(reverb_ms / 1000 * sr)
    decay      = np.exp(-np.linspace(0, 6, reverb_len))
    gain       = np.random.uniform(0.10, 0.25)
    reverb_ir  = decay * gain
    audio_rev  = np.convolve(audio, reverb_ir, mode='full')[:n]
    audio      = audio + audio_rev

    return np.clip(audio, -1.0, 1.0).astype(np.float32)

# Quick sanity check
_dummy = np.random.randn(TARGET_LEN).astype(np.float32) * 0.1
_out   = simulate_phone_recording(_dummy)
print(f"Sanity check — input RMS: {np.sqrt(np.mean(_dummy**2)):.4f}  "
      f"output RMS: {np.sqrt(np.mean(_out**2)):.4f}  shape: {_out.shape}")

In [ ]:
def extract_features(
    audio: np.ndarray,
    sr: int = SAMPLE_RATE,
    augment_spec: bool = False
) -> np.ndarray:
    feats = []

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    d_mfcc = librosa.feature.delta(mfcc)
    d2_mfcc = librosa.feature.delta(mfcc, order=2)

    n_frames = mfcc.shape[1]
    q_size = max(1, n_frames // N_MFCC_QUARTERS)

    for matrix in (mfcc, d_mfcc, d2_mfcc):
        for q in range(N_MFCC_QUARTERS):
            seg = matrix[:, q * q_size : (q + 1) * q_size]
            if seg.shape[1] == 0:
                seg = matrix[:, -1:]
            feats += list(np.mean(seg, axis=1))
            feats += list(np.std(seg,  axis=1))

    feats += list(kurtosis(mfcc, axis=1, nan_policy='omit'))
    feats += list(skew(mfcc, axis=1, nan_policy='omit'))

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    if augment_spec:
        f_mask = np.random.randint(2, 10)
        f0     = np.random.randint(0, max(1, N_MELS - f_mask))
        mel_db[f0 : f0 + f_mask, :] = mel_db.mean()
        t_mask = np.random.randint(2, 25)
        t0     = np.random.randint(0, max(1, mel_db.shape[1] - t_mask))
        mel_db[:, t0 : t0 + t_mask] = mel_db.mean()

    feats += list(np.mean(mel_db, axis=1))
    feats += list(np.std( mel_db, axis=1))

    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    feats += list(np.mean(contrast, axis=1))
    feats += list(np.std( contrast, axis=1))

    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    feats += list(np.std( chroma, axis=1))

    for feat_fn in (
        lambda: librosa.feature.zero_crossing_rate(y=audio),
        lambda: librosa.feature.spectral_centroid(y=audio, sr=sr),
        lambda: librosa.feature.spectral_rolloff(y=audio,  sr=sr),
        lambda: librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        lambda: librosa.feature.rms(y=audio),
    ):
        v = feat_fn()
        feats += [float(np.mean(v)), float(np.std(v))]

    return np.array(feats, dtype=np.float32)

_dim = extract_features(np.zeros(TARGET_LEN, dtype=np.float32)).shape[0]
print(f"Feature vector size: {_dim} dims")

In [ ]:
def augment_audio(audio: np.ndarray,
                  sr: int = SAMPLE_RATE,
                  phone_aug_prob: float = PHONE_AUG_PROB) -> list:
    n = len(audio)
    out = []

    # time-stretch
    rate = float(np.random.uniform(0.88, 1.12))
    try:
        ts = librosa.effects.time_stretch(y=audio, rate=rate)
        ts = ts[:n] if len(ts) >= n else np.pad(ts, (0, n - len(ts)))
    except Exception:
        ts = audio.copy()
    out.append(ts)

    # pitch-shift
    steps = float(np.random.uniform(-3.0, 3.0))
    try:
        ps = librosa.effects.pitch_shift(y=audio, sr=sr, n_steps=steps)
        ps = ps[:n] if len(ps) >= n else np.pad(ps, (0, n - len(ps)))
    except Exception:
        ps = audio.copy()
    out.append(ps)

    # additive noise
    noise_std = np.std(audio) / 10.0
    out.append((audio + np.random.normal(0, noise_std, n)).astype(np.float32))

    # time-shift
    shift = int(np.random.uniform(-0.15, 0.15) * sr)
    out.append(np.roll(audio, shift))

    out_final = []
    for variant in out:
        if np.random.rand() < phone_aug_prob:
            variant = simulate_phone_recording(variant, sr)
        out_final.append(variant.astype(np.float32))

    return out_final

In [ ]:
def load_icbhi_annotated(icbhi_path: str) -> dict:
    by_class = {'crackle': [], 'wheeze': [], 'normal': []}
    wav_files = sorted(f for f in os.listdir(icbhi_path) if f.endswith('.wav'))

    for wav_file in wav_files:
        wav_path = os.path.join(icbhi_path, wav_file)
        txt_path = wav_path.replace('.wav', '.txt')
        if not os.path.exists(txt_path):
            continue
        try:
            audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
            audio = normalize_loudness(audio)
        except Exception:
            continue
        try:
            ann = pd.read_csv(
                txt_path, sep='\t', header=None,
                names=['start', 'end', 'crackle', 'wheeze']
            )
        except Exception:
            continue

        for _, row in ann.iterrows():
            start_s = int(float(row['start']) * SAMPLE_RATE)
            end_s   = int(float(row['end'])   * SAMPLE_RATE)
            cycle   = audio[start_s:end_s]
            if len(cycle) / SAMPLE_RATE < MIN_CYCLE_SECS:
                continue
            cycle = prepare_clip(cycle)
            has_crackle = int(row['crackle']) == 1
            has_wheeze  = int(row['wheeze'])  == 1
            if has_crackle:
                by_class['crackle'].append(cycle)
            elif has_wheeze:
                by_class['wheeze'].append(cycle)
            else:
                by_class['normal'].append(cycle)

    return by_class

icbhi = load_icbhi_annotated(ICBHI_PATH)
for cls, clips in icbhi.items():
    print(f"  {cls:10s}: {len(clips)}")

In [ ]:
def load_snoring(snoring_path: str, max_raw: int = TARGET_PER_CLASS * 2) -> list:
    snore_dir = os.path.join(snoring_path, '1')
    files     = sorted(f for f in os.listdir(snore_dir) if f.endswith('.wav'))
    np.random.shuffle(files)
    clips = []
    for fname in files[:max_raw]:
        fpath = os.path.join(snore_dir, fname)
        try:
            audio, _ = librosa.load(fpath, sr=SAMPLE_RATE, mono=True)
            audio = normalize_loudness(audio)
            audio = prepare_clip(audio)
            snr_db    = np.random.uniform(20, 35)
            sig_rms   = np.sqrt(np.mean(audio ** 2)) + 1e-9
            noise_rms = sig_rms / (10 ** (snr_db / 20))
            audio = (audio + np.random.normal(0, noise_rms, len(audio))).astype(np.float32)
            audio = np.clip(audio, -1.0, 1.0)
            clips.append(audio)
        except Exception:
            continue
    return clips

snore_clips = load_snoring(SNORING_PATH)
print(f"snore: {len(snore_clips)} raw clips loaded")

In [ ]:
def load_other_sounds(other_path: str, max_raw: int = TARGET_PER_CLASS * 2) -> list:
    clips = []
    
    # 1. Collect structured background noise from the dataset
    bg_dir = os.path.join(other_path, '_background_noise_')
    if os.path.exists(bg_dir):
        files = sorted([f for f in os.listdir(bg_dir) if f.endswith('.wav')])
        for fname in files:
            fpath = os.path.join(bg_dir, fname)
            try:
                audio, _ = librosa.load(fpath, sr=SAMPLE_RATE, mono=True)
                audio = normalize_loudness(audio)
                
                # Split long continuous background noises into 3-second target chunks
                for i in range(0, len(audio) - TARGET_LEN + 1, TARGET_LEN):
                    if len(clips) >= max_raw // 2:
                        break
                    chunk = audio[i:i+TARGET_LEN]
                    clips.append(chunk)
            except Exception:
                continue
    
    # 2. Collect random human speech commands to teach the model to reject speech
    subdirs = sorted([d for d in os.listdir(other_path) if os.path.isdir(os.path.join(other_path, d)) and d != '_background_noise_'])
    np.random.shuffle(subdirs)
    for d in subdirs:
        if len(clips) >= max_raw:
            break
        dpath = os.path.join(other_path, d)
        files = sorted([f for f in os.listdir(dpath) if f.endswith('.wav')])
        np.random.shuffle(files)
        for fname in files[:50]: # Scrape a few from each spoken word category
            if len(clips) >= max_raw:
                break
            fpath = os.path.join(dpath, fname)
            try:
                audio, _ = librosa.load(fpath, sr=SAMPLE_RATE, mono=True)
                audio = normalize_loudness(audio)
                audio = prepare_clip(audio)
                
                # Introduce acoustic variety by blending in subtle noise
                snr_db = np.random.uniform(20, 35)
                sig_rms = np.sqrt(np.mean(audio ** 2)) + 1e-9
                noise_rms = sig_rms / (10 ** (snr_db / 20))
                audio = (audio + np.random.normal(0, noise_rms, len(audio))).astype(np.float32)
                audio = np.clip(audio, -1.0, 1.0)
                
                clips.append(audio)
            except Exception:
                continue

    return clips

other_clips = load_other_sounds(OTHER_PATH)
print(f"other: {len(other_clips)} raw clips loaded")

In [ ]:
def build_class_features(
    clips: list,
    label: str,
    target: int = TARGET_PER_CLASS
) -> tuple:
    X, y = [], []
    for clip in clips:
        if len(X) >= target:
            break
        X.append(extract_features(clip))
        y.append(label)

    aug_idx = 0
    while len(X) < target:
        base     = clips[aug_idx % len(clips)]
        variants = augment_audio(base)   # phone simulation baked in here
        for k, variant in enumerate(variants):
            if len(X) >= target:
                break
            use_spec = (k % 2 == 0)
            X.append(extract_features(variant, augment_spec=use_spec))
            y.append(label)
        aug_idx += 1

    return X[:target], y[:target]


all_X, all_y = [], []

for cls in ['crackle', 'normal', 'wheeze']:
    raw = icbhi.get(cls, [])
    if not raw:
        print(f"WARNING: No raw clips for '{cls}' — check ICBHI path.")
        continue
    print(f"Building '{cls}' ({len(raw)} raw → {TARGET_PER_CLASS}) …")
    Xc, yc = build_class_features(raw, cls, TARGET_PER_CLASS)
    all_X.extend(Xc)
    all_y.extend(yc)

print(f"Building 'snore'  ({len(snore_clips)} raw → {TARGET_PER_CLASS}) …")
Xs, ys = build_class_features(snore_clips, 'snore', TARGET_PER_CLASS)
all_X.extend(Xs)
all_y.extend(ys)

print(f"Building 'other'  ({len(other_clips)} raw → {TARGET_PER_CLASS}) …")
Xo, yo = build_class_features(other_clips, 'other', TARGET_PER_CLASS)
all_X.extend(Xo)
all_y.extend(yo)

X = np.array(all_X, dtype=np.float32)
y = np.array(all_y)
print(f"\nFull dataset : {X.shape}")
print("Class counts:", dict(Counter(y)))

In [ ]:
def mixup_features(
    X: np.ndarray,
    y: np.ndarray,
    n_extra_per_class: int = 200,
    alpha: float = 0.3
) -> tuple:
    classes  = np.unique(y)
    X_new, y_new = [], []
    for cls in classes:
        mask  = (y == cls)
        X_cls = X[mask]
        for _ in range(n_extra_per_class):
            i, j = np.random.choice(len(X_cls), 2, replace=False)
            lam  = np.random.beta(alpha, alpha)
            X_new.append(lam * X_cls[i] + (1 - lam) * X_cls[j])
            y_new.append(cls)
    X_out = np.vstack([X, np.array(X_new, dtype=np.float32)])
    y_out = np.concatenate([y, np.array(y_new)])
    return X_out, y_out

In [ ]:
encoder = LabelEncoder()
y_enc   = encoder.fit_transform(y)
print("Classes:", list(encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)
print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")

X_train, y_train = mixup_features(X_train, y_train, n_extra_per_class=200)
print(f"Train after mixup : {X_train.shape}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [ ]:
def lgbm_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators':      trial.suggest_int(  'n_estimators',      200, 1000),
        'learning_rate':     trial.suggest_float('learning_rate',      0.01, 0.15, log=True),
        'max_depth':         trial.suggest_int(  'max_depth',          4,   12),
        'num_leaves':        trial.suggest_int(  'num_leaves',         20,  150),
        'min_child_samples': trial.suggest_int(  'min_child_samples',  10,  60),
        'subsample':         trial.suggest_float('subsample',          0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha',          1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda',         1e-3, 10.0, log=True),
        'class_weight':      'balanced',
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
    }
    model  = lgb.LGBMClassifier(**params)
    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_train_sc, y_train,
        cv=skf, scoring='accuracy', n_jobs=1
    )
    return float(scores.mean())

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(lgbm_objective, n_trials=100, show_progress_bar=True)

best_lgbm_params = study.best_params
best_lgbm_params.update({
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs':       -1,
    'verbose':      -1
})
print(f"\nBest CV accuracy: {study.best_value:.4f}")
print("Best params:", best_lgbm_params)

In [ ]:
lgbm_model = lgb.LGBMClassifier(**best_lgbm_params)
lgbm_model.fit(X_train_sc, y_train)
print(f"LightGBM  train acc: {accuracy_score(y_train, lgbm_model.predict(X_train_sc)):.4f}")
print(f"LightGBM  test  acc: {accuracy_score(y_test,  lgbm_model.predict(X_test_sc)):.4f}")

In [ ]:
svm_model = SVC(
    kernel='rbf', C=10, gamma='scale',
    probability=True, class_weight='balanced', random_state=42
)
svm_model.fit(X_train_sc, y_train)
print(f"SVM  test acc: {accuracy_score(y_test, svm_model.predict(X_test_sc)):.4f}")

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=600, max_depth=None, min_samples_leaf=4,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train_sc, y_train)
print(f"RF test acc : {accuracy_score(y_test, rf_model.predict(X_test_sc)):.4f}")

In [ ]:
ensemble = VotingClassifier(
    estimators=[('lgbm', lgbm_model), ('svm', svm_model), ('rf', rf_model)],
    voting='soft'
)
ensemble.fit(X_train_sc, y_train)

y_pred = ensemble.predict(X_test_sc)
final_acc = accuracy_score(y_test, y_pred)
print(f"\nEnsemble test accuracy : {final_acc:.4f} ({final_acc*100:.1f}%)")

In [ ]:
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_,
    ax=ax
)
ax.set_title(f'Ensemble Confusion Matrix — Test Accuracy: {final_acc:.4f} (v4)',
             fontsize=13, pad=12)
ax.set_ylabel('True Label',      fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix_v4.png', dpi=150)
plt.show()

In [ ]:
for name, model in [
    ('LightGBM',     lgbm_model),
    ('SVM (RBF)',    svm_model),
    ('RandomForest', rf_model),
    ('Ensemble',     ensemble),
]:
    tr = accuracy_score(y_train, model.predict(X_train_sc))
    te = accuracy_score(y_test,  model.predict(X_test_sc))
    print(f"{name:<18} {tr:>10.4f} {te:>10.4f} {tr-te:>8.4f}")

In [ ]:
skf_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    lgb.LGBMClassifier(**best_lgbm_params),
    scaler.transform(X),
    y_enc,
    cv=skf_full,
    scoring='accuracy',
    n_jobs=1
)
print(f"5-fold CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold: {[round(s, 4) for s in cv_scores]}")

In [ ]:
for obj, fname in [
    (ensemble,   'ensemble_v4.pkl'),
    (lgbm_model, 'lgbm_v4.pkl'),
    (scaler,     'scaler_v4.pkl'),
    (encoder,    'encoder_v4.pkl'),
]:
    with open(fname, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    print(f"Saved  {fname}")